In [ ]:
!pip install datasets scanpy anndata pandas pyarrow nbstripout

1. Load Metadata Mapping

In [ ]:
import pandas as pd

# The Orion dataset for HEK293T is split into two shards
data_urls = [
    "https://huggingface.co/api/datasets/Xaira-Therapeutics/X-Atlas-Orion/parquet/default/HEK293T/0.parquet",
    "https://huggingface.co/api/datasets/Xaira-Therapeutics/X-Atlas-Orion/parquet/default/HEK293T/1.parquet"
]

# Professional workflow: Load shards into a list and concatenate them
# This is much faster and cleaner than manual merging
print("Loading data shards...")
data_shards = [pd.read_parquet(url) for url in data_urls]
combined_data = pd.concat(data_shards, ignore_index=True)

print(f"Handshake successful. Total cells loaded: {len(combined_data)}")

# Verify the schema you identified earlier (gene_token_id, gene_expression)
print(combined_data[['gene_token_id', 'gene_expression']].head())

2. Deserialisation

In [ ]:
import pickle

# Load the 35 filtered gout cells
with open('gout-transcriptome-causal/data/gout_cells_subset.pkl', 'rb') as f:
    gout_cells = pickle.load(f)

print(f"Deserialized {len(gout_cells)} target cells for processing.")

3. Rank-Value Encoding

In [ ]:
def preprocess_for_geneformer(cell_data, max_len=2048):
    """
    Transforms raw counts into rank-ordered Geneformer tokens.
    """
    # 1. Extract raw identifiers and counts
    tokens = cell_data['gene_token_id']
    counts = cell_data['gene_expression']

    # 2. Filter for non-zero expression and sort descending (Rank-Ordering)
    # This creates the "causal grammar" the model expects
    temp_df = pd.DataFrame({'token': tokens, 'count': counts})
    temp_df = temp_df[temp_df['count'] > 0].sort_values(by='count', ascending=False)

    # 3. Truncate to Geneformer's standard context length
    return temp_df['token'].tolist()[:max_len]

# Tokenize your gout dataset
tokenized_cells = [preprocess_for_geneformer(cell) for cell in gout_cells]

print(f"Cell 1 Example (Top 5 Tokens): {tokenized_cells[0][:5]}")